%md
# 03 – Variant A (baseline RAG)
Retrieves the top 3 policy chunks for a question and asks the LLM to answer using them. No citations, no refusal instructions, no hard rules. Those are added in Variant B.


In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
CATALOG = "workspace"
SCHEMA = "policy_rag"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.policy_chunks_index"
ENDPOINT_NAME = "policy_rag_endpoint"

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
TOP_K = 3            # number of chunks retrieved per question
TEMPERATURE = 0.0    # 0 = as deterministic as possible, important for reproducibility
MAX_TOKENS = 500     # max length of the answer

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

vsc = VectorSearchClient(disable_notice=True)
index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
w = WorkspaceClient()

In [0]:
def retrieve(question, k=TOP_K):
    """Return the top-k chunks for a question as a list of dicts."""
    results = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "document", "section", "text"],
        num_results=k,
    )
    return [
        {"chunk_id": chunk_id, "document": document, "section": section, "text": text, "score": score}
        for chunk_id, document, section, text, score in results["result"]["data_array"]
    ]

In [0]:
SYSTEM_PROMPT_A = (
    "You are an assistant that answers employee questions about the internal policies "
    "of Nordljus Consulting AB. Use the context below to answer the question."
)

def build_messages(question, chunks):
    context = "\n\n---\n\n".join(chunk["text"] for chunk in chunks)
    user_content = f"Context:\n{context}\n\nQuestion: {question}"
    return [
        ChatMessage(role=ChatMessageRole.SYSTEM, content=SYSTEM_PROMPT_A),
        ChatMessage(role=ChatMessageRole.USER, content=user_content),
    ]

In [0]:
def answer_variant_a(question):
    chunks = retrieve(question)
    response = w.serving_endpoints.query(
        name=LLM_ENDPOINT,
        messages=build_messages(question, chunks),
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return {
        "variant": "A",
        "question": question,
        "retrieved_chunk_ids": [c["chunk_id"] for c in chunks],
        "retrieved_scores": [round(c["score"], 3) for c in chunks],
        "answer": response.choices[0].message.content,
    }

In [0]:
test_questions = [
    "How many paid vacation days do employees at Nordljus Consulting AB have?",
    "How many paid vacation days do employees in Sweden get?",
    "Can I stay home when my kid is sick?",
    "Can I paste a customer contract into Microsoft 365 Copilot?",
    "What is the company's pension plan?",
]

for q in test_questions:
    result = answer_variant_a(q)
    print(f"Q: {result['question']}")
    print(f"Retrieved: {result['retrieved_chunk_ids']}")
    print(f"A: {result['answer']}")
    print("-" * 80)